<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/marco-canas/fundamentos_de_programacion/blob/main/2_clases/unidad4/11_chapter_matplotlib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/marco-canas/fundamentos_de_programacion/blob/main/2_clases/unidad4/11_chapter_matplotlib.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Parcial 4 

### Habilidades a evaluar sobre Gestión de Bases de Datos con SQL

Saber manejar a cabalidad la sentencia `SELECT`

# Taller práctico en SQL  

Este es un script de Python completo para consolidar tus archivos `.csv` limpios dentro de una base de datos relacional SQLite (`.db`).



El script está diseñado para buscar automáticamente los archivos en el directorio que especificaste (`C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\2_clases\unidad4\2_datos\`), aplicarles de forma rápida el **pipeline de limpieza** que estructuramos en los puntos anteriores para evitar guardar datos corruptos, y finalmente guardarlos como tablas independientes dentro de un único archivo llamado `monitoreo_y_gestion.db`.


In [1]:
import os
import sqlite3
import pandas as pd

# 1. Definir la ruta del directorio de trabajo
directorio_datos = r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\3_paciales\4_parcial\2_datos"

# 2. Definir los nombres de los 6 archivos CSV y sus respectivas tablas SQL
archivos_csv = {
    "rendimiento_estudiantes.csv": "rendimiento_estudiantes",
    "ventas_tienda_retail.csv": "ventas_retail",
    "uso_plataforma_streaming.csv": "uso_streaming",
    "indicadores_salud_hospital.csv": "indicadores_salud",
    "desempeño_empleados_rrhh.csv": "desempeño_rrhh",
    "contaminacion_ambiental_ciudades.csv": "contaminacion_ambiental"  # <- AGREGADO: Sexto archivo
}

# 3. Ruta de la base de datos SQL
ruta_base_datos = os.path.join(directorio_datos, "monitoreo_y_gestion.db")

print(f"Iniciando la creación de la base de datos en: {ruta_base_datos}\n")

try:
    # 4. Establecer conexión con SQLite
    conexion = sqlite3.connect(ruta_base_datos)
    cursor = conexion.cursor()

    # 5. Iterar sobre cada archivo para limpiarlo y migrarlo a SQL
    for archivo, nombre_tabla in archivos_csv.items():
        ruta_completa_csv = os.path.join(directorio_datos, archivo)

        # Verificar si el archivo CSV realmente existe en el disco
        if not os.path.exists(ruta_completa_csv):
            print(f"Advertencia: No se encontró el archivo '{archivo}'. Se omitirá.")
            continue

        print(f"Procesando '{archivo}'...")
        df = pd.read_csv(ruta_completa_csv)

        # ---------------------------------------------------------------------
        # PIPELINE DE SANEAMIENTO REVISADO (Protección estricta de dtypes)
        # ---------------------------------------------------------------------
        
        # Caso 1: Estudiantes
        if nombre_tabla == "rendimiento_estudiantes":
            df["Nota_Final"] = df["Nota_Final"].astype(float)
            df["Asistencia_Porcentaje"] = df["Asistencia_Porcentaje"].astype(float)
            
            df.loc[df["Nota_Final"] > 5.0, "Nota_Final"] = df.loc[df["Nota_Final"] <= 5.0, "Nota_Final"].median()
            df.loc[df["Horas_Estudio_Semana"] < 0, "Horas_Estudio_Semana"] = 0
            df["Asistencia_Porcentaje"] = df["Asistencia_Porcentaje"].fillna(df["Asistencia_Porcentaje"].median())

        # Caso 2: Ventas Retail
        elif nombre_tabla == "ventas_retail":
            df = df.drop_duplicates()
            df["Precio_Unitario"] = df["Precio_Unitario"].astype(float)
            
            df["Precio_Unitario"] = df.groupby("Categoria_Producto")["Precio_Unitario"].transform(lambda x: x.fillna(x.median()))
            df.loc[df["Cantidad"] > 100, "Cantidad"] = df.loc[df["Cantidad"] <= 100, "Cantidad"].median()
            df["Ingresos_Totales"] = df["Cantidad"] * df["Precio_Unitario"]

        # Caso 3: Streaming
        elif nombre_tabla == "uso_streaming":
            df["Edad"] = df["Edad"].astype(float)
            df["Minutos_Vistos_Mes"] = df["Minutos_Vistos_Mes"].astype(float)
            
            df.loc[df["Edad"] > 100, "Edad"] = df.loc[df["Edad"] <= 100, "Edad"].mean()
            df["Tipo_Suscripcion"] = df["Tipo_Suscripcion"].replace("PREMIUM_ERR", "Premium")
            df.loc[df["Minutos_Vistos_Mes"] < 0, "Minutos_Vistos_Mes"] = 0.0

        # Caso 4: Salud Hospital
        elif nombre_tabla == "indicadores_salud":
            df["Presion_Sistolica"] = df["Presion_Sistolica"].astype(float)
            df["Colesterol_mgDl"] = df["Colesterol_mgDl"].astype(float)
            
            df.loc[df["Presion_Sistolica"] < 40, "Presion_Sistolica"] = df.loc[df["Presion_Sistolica"] >= 40, "Presion_Sistolica"].median()
            df["Colesterol_mgDl"] = df.groupby("Diagnostico_Riesgo")["Colesterol_mgDl"].transform(lambda x: x.fillna(x.median()))

        # Caso 5: Recursos Humanos
        elif nombre_tabla == "desempeño_rrhh":
            df["Salario_Mensual_USD"] = df["Salario_Mensual_USD"].astype(float)
            df["Puntuacion_Evaluacion"] = df["Puntuacion_Evaluacion"].astype(float)
            
            df.loc[df["Salario_Mensual_USD"] > 15000, "Salario_Mensual_USD"] = 7500.0
            df["Puntuacion_Evaluacion"] = df.groupby("Departamento")["Puntuacion_Evaluacion"].transform(lambda x: x.fillna(x.mean()))

        # Caso 6: Contaminación Ambiental  <- AGREGADO: Bloque lógico de curación
        elif nombre_tabla == "contaminacion_ambiental":
            # 1. Purgar filas donde la variable ancla (Ciudad) sea nula
            df = df.dropna(subset=["Ciudad"])
            
            # 2. Forzar variables térmicas y físicas a float64
            df["Temperatura_C"] = df["Temperatura_C"].astype(float)
            df["Nivel_PM25"] = df["Nivel_PM25"].astype(float)
            df["Humedad_Porcentaje"] = df["Humedad_Porcentaje"].astype(float)
            
            # 3. Corregir el outlier de telemetría de -40.0°C usando la mediana local
            # Nota: Agrupamos de forma segura buscando los datos de la ciudad con problemas térmicos
            condicion_error = df["Temperatura_C"] == -40.0
            if condicion_error.any():
                df["Temperatura_C"] = df.groupby("Ciudad")["Temperatura_C"].transform(
                    lambda x: x.replace(-40.0, x[x > -40.0].median())
                )

        # ---------------------------------------------------------------------
        # MIGRACIÓN A LA BASE DE DATOS
        # ---------------------------------------------------------------------
        df.to_sql(nombre_tabla, conexion, if_exists="replace", index=False)
        print(f"   ¡Éxito! Insertadas {len(df)} filas en la tabla SQL '{nombre_tabla}'.")

    conexion.commit()
    print("\n" + "=" * 60)
    print(" BASE DE DATOS SQLITE CREADA Y CONSOLIDADA CON ÉXITO (6 TABLAS)")
    print("=" * 60)

except sqlite3.Error as error:
    print(f"Ocurrió un error al manipular SQLite: {error}")

finally:
    if conexion:
        conexion.close()
        print("Conexión con la base de datos cerrada de forma segura.")

Iniciando la creación de la base de datos en: C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\3_paciales\4_parcial\2_datos\monitoreo_y_gestion.db

Procesando 'rendimiento_estudiantes.csv'...
   ¡Éxito! Insertadas 120 filas en la tabla SQL 'rendimiento_estudiantes'.
Procesando 'ventas_tienda_retail.csv'...
   ¡Éxito! Insertadas 200 filas en la tabla SQL 'ventas_retail'.
Procesando 'uso_plataforma_streaming.csv'...
   ¡Éxito! Insertadas 150 filas en la tabla SQL 'uso_streaming'.
Procesando 'indicadores_salud_hospital.csv'...
   ¡Éxito! Insertadas 100 filas en la tabla SQL 'indicadores_salud'.
Procesando 'desempeño_empleados_rrhh.csv'...
   ¡Éxito! Insertadas 80 filas en la tabla SQL 'desempeño_rrhh'.
Procesando 'contaminacion_ambiental_ciudades.csv'...
   ¡Éxito! Insertadas 175 filas en la tabla SQL 'contaminacion_ambiental'.

 BASE DE DATOS SQLITE CREADA Y CONSOLIDADA CON ÉXITO (6 TABLAS)
Conexión con la base de datos cerrada de forma segura.



# Ventajas Pedagógicas de esta Solución:



1. **Unificación del Entorno de Datos:** En lugar de hacer que carguen de forma independiente múltiples archivos planos por toda la computadora, ahora solo deben conectarse a `monitoreo_y_gestion.db`.


2. **Introducción Práctica a SQL en Python:** Este script prepara la clase para introducir librerías de conexión relacional o para realizar consultas directas con Pandas usando comandos como:



In [3]:
import pandas as pd 
import sqlite3 
conexion = sqlite3.connect(r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\3_paciales\4_parcial\2_datos\monitoreo_y_gestion.db")
df_ventas = pd.read_sql_query(
    "SELECT * FROM ventas_retail WHERE Cantidad > 3", conexion
)
df_ventas.head()

,ID_Transaccion,Fecha,Categoria_Producto,Cantidad,Precio_Unitario,Metodo_Pago,Ingresos_Totales
0,TX-0015,2026-01-15,Alimentos,5,209.33,Efectivo,1046.65
1,TX-0021,2026-01-17,Ropa,4,6.16,Efectivo,24.64
2,TX-0028,2026-01-10,Electrónica,4,396.06,Efectivo,1584.24
3,TX-0038,2026-01-13,Ropa,4,490.39,Transferencia,1961.56
4,TX-0056,2026-01-04,Hogar,4,373.81,Tarjeta,1495.24




3. **Persistencia Garantizada:** Al usar `if_exists='replace'`, si algún estudiante corrompe los datos en memoria mientras practica, solo necesita volver a correr este script para restaurar las tablas limpias de fábrica en un segundo.

# Asignación Wilfran 

Diseño para el **Parcial N° 1 de Fundamentos de Bases de Datos (SQL)**, aplicado exclusivamente a la Tabla 1 (`rendimiento_estudiantes`) de la base de datos relacional `monitoreo_y_gestion.db` que acabamos de estructurar.



Siguiendo tu enfoque pedagógico, el examen tiene una **curva de aprendizaje progresiva** (va sumando una cláusula SQL por cada punto) y está estructurado para que los estudiantes escriban el código SQL y argumenten su interpretación en celdas de tipo *Markdown*.



# PARCIAL 1: FUNDAMENTOS DE CONSULTAS SQL

**Asignatura:** Fundamentos de Programación / Bases de Datos

**Profesor:** Marco Julio Cañas Campillo

**Base de Datos Relacional:** `monitoreo_y_gestion.db`

**Tabla de Trabajo:** `rendimiento_estudiantes`




### Contexto del Examen: Esquema de la Tabla

Antes de iniciar, recuerda que la tabla `rendimiento_estudiantes` cuenta con los siguientes campos (*columnas*):

* `ID_Estudiante` (Texto)
* `Género` (Texto)
* `Horas_Estudio_Semana` (Entero)
* `Asistencia_Porcentaje` (Entero)
* `Nota_Final` (Decimal)
* `Socioeconómico` (Texto: Bajo, Medio, Alto)

---



# Punto 1: Estructura Básica de Proyección (10 Puntos)
[Video de apoyo para el punto 1](https://www.youtube.com/watch?v=w_Ti7g2NIRw)
**Sentencias a evaluar:** `SELECT` y `FROM`

**Enunciado:** La oficina de Registro Académico necesita un listado inicial con la información de contacto e identificación de todo el alumnado de la muestra. Escribe una consulta en SQL que extraiga **únicamente** las columnas correspondientes al identificador del estudiante, su género y su nivel socioeconómico de toda la tabla.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** ¿Qué diferencia técnica o de rendimiento existe entre hacer la consulta especificando los nombres de los campos frente al uso del comodín asterisco (`SELECT * FROM...`)? Justifica tu respuesta.




In [5]:
import pandas as pd 
import sqlite3 
conexion = sqlite3.connect(r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\3_paciales\4_parcial\2_datos\monitoreo_y_gestion.db")
solicitud = """
SELECT ID_Estudiante, Género, Socioeconómico 
FROM rendimiento_estudiantes
""" 
df = pd.read_sql_query(solicitud, conexion)
df 


,ID_Estudiante,Género,Socioeconómico
0,EST-001,M,Medio
1,EST-002,No Binario,Alto
2,EST-003,F,Alto
3,EST-004,F,Medio
4,EST-005,M,Medio
...,...,...,...
115,EST-116,F,Medio
116,EST-117,F,Medio
117,EST-118,M,Bajo
118,EST-119,F,Bajo



# Punto 2: Filtrado Condicional de Registros (10 Puntos)
[Video de apoyo para este segundo punto](https://www.youtube.com/watch?v=3Ihb6Q9tgTA)
**Sentencias a evaluar:** `SELECT`, `FROM` y `WHERE`

**Enunciado:** El departamento de Bienestar Universitario quiere identificar a los estudiantes en situación de vulnerabilidad o riesgo de reprobación.   
Escribe una consulta en SQL que devuelva el `ID_Estudiante`, las `Horas_Estudio_Semana` y la `Nota_Final` **solo** para aquellos alumnos cuyo nivel `Socioeconómico` sea igual a 'Bajo' **y** que además tengan una `Nota_Final` inferior a 3.0.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Explica el papel que desempeña el operador lógico `AND` dentro de tu cláusula `WHERE` en este escenario y qué ocurriría con los resultados si lo cambiaras por un operador `OR`.



In [ ]:
solicitud = """
SELECT ID_Estudiante, Horas_Estudio_Semana, Nota_Final, Socioeconómico, Nota_Final
FROM rendimiento_estudiantes
WHERE Socioeconómico = 'Bajo' AND Nota_Final < 3.0
"""

df = pd.read_sql_query(solicitud, conexion)
df 

,ID_Estudiante,Horas_Estudio_Semana,Nota_Final,Socioeconómico,Nota_Final
0,EST-011,10,1.2,Bajo,1.2
1,EST-015,33,2.9,Bajo,2.9
2,EST-016,33,1.4,Bajo,1.4
3,EST-025,24,2.5,Bajo,2.5
4,EST-033,7,1.1,Bajo,1.1
5,EST-049,4,1.4,Bajo,1.4
6,EST-054,20,1.1,Bajo,1.1
7,EST-071,2,2.4,Bajo,2.4
8,EST-084,9,1.0,Bajo,1.0
9,EST-094,5,2.3,Bajo,2.3


In [7]:
(5/6)*5

4.166666666666667


# Punto 3: Agregación y Resumen de Datos (15 Puntos)
[Video de apoyo al punto 3](https://www.youtube.com/watch?v=L6X4w9g4A1Y) 
**Sentencias a evaluar:** `SELECT`, `FROM`, `WHERE` y `GROUP BY`

**Enunciado:** La decanatura de la facultad está realizando un estudio comparativo del rendimiento académico. Escribe una consulta en SQL que muestre el `Género` y la **nota final promedio** (utiliza la función de agregación `AVG()`) de los estudiantes, agrupados por su género. Sin embargo, para evitar sesgos por inasistencia, la consulta debe incluir **únicamente** a los estudiantes que tengan una `Asistencia_Porcentaje` mayor o igual a 80.

* **Celda de Código SQL:**


In [ ]:

import pandas as pd
import sqlite3

conexion = sqlite3.connect(r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\3_paciales\4_parcial\2_datos\monitoreo_y_gestion.db")
solicitud = """
SELECT Género, AVG(Nota_Final) AS Promedio_Nota_Final
FROM rendimiento_estudiantes
GROUP BY Género
"""

df = pd.read_sql_query(solicitud, conexion)
df 


* **Celda Markdown (Pregunta de Análisis):** ¿Por qué es obligatorio incluir la columna `Género` tanto en la instrucción `SELECT` como en la instrucción `GROUP BY`? ¿Qué error de consistencia matemática arrojaría el motor de base de datos si la omitieras en el ordenamiento?



In [20]:
import pandas as pd
import sqlite3
conexion = sqlite3.connect(r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\3_paciales\4_parcial\2_datos\monitoreo_y_gestion.db")

solicitud = """
SELECT Género, AVG(Nota_Final) AS Promedio_Nota
FROM rendimiento_estudiantes
WHERE Asistencia_Porcentaje >= 80
GROUP BY Género
"""
df = pd.read_sql_query(solicitud, conexion)
df

,Género,Promedio_Nota
0,F,2.710345
1,M,3.143478
2,No Binario,3.000000



# Punto 4: Restricción y Ranking de Resultados (15 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `ORDER BY` y `LIMIT`

**Enunciado:** El comité de becas de excelencia académica desea encontrar los mejores promedios de horas de estudio invertidas en los estratos más necesitados. Escribe una consulta en SQL que agrupe a los estudiantes por su nivel `Socioeconómico` y calcule el promedio de `Horas_Estudio_Semana`. Aplica las siguientes restricciones en orden:

1. Filtra para que solo se evalúe a los estudiantes con una `Nota_Final` sobresaliente (mayor o igual a 4.0).
2. Ordena los resultados de manera descendente (`DESC`) según el promedio de horas calculado.
3. Limita la salida para mostrar **únicamente el primer registro** (el estrato con el promedio de estudio más alto).


In [16]:

# * **Celda de Código SQL:**

#```sql
#-- Escribe tu consulta aquí
solicitud = """  
SELECT 
    Socioeconómico, 
    AVG(Horas_Estudio_Semana) AS Promedio_Horas_Estudio
FROM 
    rendimiento_estudiantes
WHERE 
    Nota_Final >= 4.0
GROUP BY 
    Socioeconómico
ORDER BY 
    Promedio_Horas_Estudio DESC
LIMIT 3;
""" 

df = pd.read_sql_query(solicitud, conexion)
df

,Socioeconómico,Promedio_Horas_Estudio
0,Alto,18.555556
1,Bajo,18.300000
2,Medio,14.000000



* **Celda Markdown (Pregunta de Análisis):** Describe detalladamente cuál es el **orden lógico de ejecución** que sigue internamente el motor SQL para resolver esta consulta (es decir, qué cláusula se procesa primero en la memoria, cuál segunda, y así sucesivamente hasta el `LIMIT`).




# Punto 5: Reto Práctico Integrador en Python / Pandas (Opcional / Bono de Clase)

**Enunciado:** Abre una celda de código en tu cuaderno de Jupyter y escribe el script de Python necesario para conectarte a la base de datos `monitoreo_y_gestion.db` (usando la librería `sqlite3`). Carga mediante la función `pd.read_sql_query()` la consulta exacta que desarrollaste en el **Punto 3** (Escribe una consulta en SQL que muestre el `Género` y la **nota final promedio** (utiliza la función de agregación `AVG()`) de los estudiantes, agrupados por su género. Sin embargo, para evitar sesgos por inasistencia, la consulta debe incluir **únicamente** a los estudiantes que tengan una `Asistencia_Porcentaje` mayor o igual a 80.
) y almacénala en un DataFrame llamado `df_resultado`. Imprime las filas obtenidas en pantalla.

* **Celda de Código Python:**


In [17]:
# Desarrolla el bloque de conexión y carga con Pandas aquí

import pandas as pd  
import sqlite3
conexion = sqlite3.connect(r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\3_paciales\4_parcial\2_datos\monitoreo_y_gestion.db")

solicitud = """
SELECT Género, AVG(Nota_Final) AS Nota_Final_Promedio
FROM rendimiento_estudiantes
WHERE Asistencia_Porcentaje >= 80.0
GROUP BY Género
"""

df_resultado = pd.read_sql_query(solicitud, conexion)
df_resultado

,Género,Nota_Final_Promedio
0,F,2.710345
1,M,3.143478
2,No Binario,3.000000



# Hoja de Respuestas Esperadas (Para el Profesor):

* **Solución Punto 1:** ```sql
SELECT ID_Estudiante, Género, Socioeconómico FROM rendimiento_estudiantes;
```



```


* **Solución Punto 2:** ```sql
SELECT ID_Estudiante, Horas_Estudio_Semana, Nota_Final
FROM rendimiento_estudiantes
WHERE Socioeconómico = 'Bajo' AND Nota_Final < 3.0;
```



```


* **Solución Punto 3:** ```sql
SELECT Género, AVG(Nota_Final) AS Nota_Promedio
FROM rendimiento_estudiantes
WHERE Asistencia_Porcentaje >= 80
GROUP BY Género;
```



```


* **Solución Punto 4:** ```sql
SELECT Socioeconómico, AVG(Horas_Estudio_Semana) AS Horas_Promedio
FROM rendimiento_estudiantes
WHERE Nota_Final >= 4.0
GROUP BY Socioeconómico
ORDER BY Horas_Promedio DESC
LIMIT 1;
```



```


* **Solución Punto 5 (Python):** ```python
import sqlite3
import pandas as pd
conn = sqlite3.connect(
r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\2_clases\unidad4\2_datos\monitoreo_y_gestion.db"
)
query = "SELECT Género, AVG(Nota_Final) FROM rendimiento_estudiantes WHERE Asistencia_Porcentaje >= 80 GROUP BY Género;"
df_resultado = pd.read_sql_query(query, conn)
print(df_resultado)
conn.close()
```


# Asignación Pablo  

Aquí tienes el diseño para el **Parcial N° 2 de Fundamentos de Bases de Datos (SQL)**, aplicado de manera exclusiva a la Tabla 2 (`ventas_retail`) de la base de datos relacional `monitoreo_y_gestion.db`.

Al igual que el examen anterior, este parcial mantiene una **curva de aprendizaje estrictamente progresiva**, acumulando una cláusula SQL nueva en cada punto para evaluar la madurez técnica y el criterio de analítica de negocios (*Business Analytics*) de los estudiantes.




# PARCIAL 2: EXTRACCIÓN Y ANALÍTICA DE DATOS COMERCIALES CON SQL

**Asignatura:** Fundamentos de Programación / Bases de Datos

**Profesor:** Marco Cañas

**Base de Datos Relacional:** `monitoreo_y_gestion.db`

**Tabla de Trabajo:** `ventas_retail`



### Contexto del Examen: Esquema de la Tabla

La tabla `ventas_retail` registra el histórico de transacciones de la tienda y cuenta con las siguientes columnas:

* `ID_Transaccion` (Texto)
* `Fecha` (Texto/Date)
* `Categoria_Producto` (Texto: Electrónica, Ropa, Hogar, Alimentos)
* `Cantidad` (Entero)
* `Precio_Unitario` (Decimal)
* `Metodo_Pago` (Texto: Tarjeta, Efectivo, Transferencia)
* `Ingresos_Totales` (Decimal: Resultado de Cantidad $\times$ Precio_Unitario)

---



## Punto 1: Proyección de Atributos de Inventario (10 Puntos)

**Sentencias a evaluar:** `SELECT` y `FROM`

**Enunciado:** El equipo de auditoría interna de la tienda requiere un reporte simplificado para verificar el catálogo de precios cargado en el sistema. Escribe una consulta en SQL que extraiga **únicamente** las columnas correspondientes a la categoría del producto, el precio unitario y el método de pago de todos los registros de la tabla.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Si la tabla tuviera 10 millones de filas y solo necesitas analizar los métodos de pago, ¿por qué se considera una mala práctica de ingeniería de datos escribir `SELECT * FROM ventas_retail`? Explica desde la perspectiva del uso de memoria y ancho de banda.

---



# Punto 2: Filtrado de Transacciones de Alto Valor (10 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM` y `WHERE`

**Enunciado:** El departamento de seguridad y prevención de fraudes necesita supervisar las transacciones de montos elevados realizadas en efectivo para cumplir con las regulaciones locales. Escribe una consulta en SQL que devuelva el `ID_Transaccion`, la `Fecha`, la `Categoria_Producto` y los `Ingresos_Totales` **solo** para aquellas ventas de la categoría 'Electrónica' **cuya** vía de pago sea 'Efectivo' **y** cuyos `Ingresos_Totales` superen los 1000.0 dólares.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** En la cláusula `WHERE`, ¿qué diferencia matemática y de resultado existiría si por error el estudiante escribe `Ingresos_Totales >= 1000` en lugar de `Ingresos_Totales > 1000`? Justifica la importancia de los operadores relacionales estrictos en auditoría financiera.



# Punto 3: Consolidación y Agregación por Unidades de Negocio (15 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM`, `WHERE` y `GROUP BY`

**Enunciado:** La gerencia financiera está preparando el cierre contable mensual y desea analizar la liquidez proveniente de pagos digitales. Escribe una consulta en SQL que muestre la `Categoria_Producto` y la **suma total de ingresos generados** (utiliza la función de agregación `SUM(Ingresos_Totales)` renombrada como `Ventas_Totales`) agrupada por cada categoría. Condiciona la consulta para que incluya **únicamente** transacciones donde el `Metodo_Pago` haya sido 'Tarjeta' o 'Transferencia'.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Si ejecutas una función de agregación como `SUM()` o `AVG()` junto a una columna normal (como `Categoria_Producto`), ¿qué función cumple la sentencia `GROUP BY` para evitar que el motor relacional genere un error de inconsistencia en las dimensiones de la tabla resultante?




# Punto 4: Reporte de Rendimiento y Análisis de Competitividad (15 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `ORDER BY` y `LIMIT`

**Enunciado:** El director de compras de la cadena de tiendas desea identificar cuál es el producto o categoría líder que registra el ticket o precio promedio de venta más alto, evaluando transacciones de volumen regular. Escribe una consulta en SQL que agrupe los datos por `Categoria_Producto` y calcule el **precio unitario promedio** (usa la función `AVG(Precio_Unitario)`). Aplica los siguientes parámetros:

1. Filtra mediante `WHERE` para evaluar únicamente operaciones donde la `Cantidad` de artículos comprados sea menor o igual a 3 unidades (evitando compras mayoristas atípicas).
2. Ordena los resultados de manera descendente (`DESC`) según el promedio del precio unitario calculado.
3. Restringe el resultado utilizando `LIMIT` para extraer **únicamente la categoría con el promedio más alto**.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Durante el procesamiento de esta consulta, el motor de la base de datos aplica el filtro `WHERE` **antes** de realizar la agrupación con `GROUP BY`. ¿Qué beneficio tiene este orden lógico de ejecución en términos de optimización y velocidad de cálculo en comparación con filtrar los datos después de agruparlos?




# Punto 5: Automatización de Reportes Gerenciales con Pandas (Bono de Clase)

**Enunciado:** Abre una celda de código en tu cuaderno de Jupyter y desarrolla el script de Python necesario para conectarte a la base de datos `monitoreo_y_gestion.db` (usando `sqlite3`). Extrae el reporte estructurado en el **Punto 4** directamente a un DataFrame de Pandas llamado `df_lider_ventas` e imprime el resultado final formateado.

* **Celda de Código Python:**

```python
# Desarrolla el bloque de código de extracción aquí

```




# Hoja de Respuestas Esperadas (Para uso exclusivo del Profesor):

* **Solución Punto 1:**
```sql
SELECT Categoria_Producto, Precio_Unitario, Metodo_Pago FROM ventas_retail;

```


* **Solución Punto 2:**
```sql
SELECT ID_Transaccion, Fecha, Categoria_Producto, Ingresos_Totales 
FROM ventas_retail 
WHERE Categoria_Producto = 'Electrónica' 
  AND Metodo_Pago = 'Efectivo' 
  AND Ingresos_Totales > 1000.0;

```


* **Solución Punto 3:**
```sql
SELECT Categoria_Producto, SUM(Ingresos_Totales) AS Ventas_Totales 
FROM ventas_retail 
WHERE Metodo_Pago = 'Tarjeta' OR Metodo_Pago = 'Transferencia' -- También es válido: WHERE Metodo_Pago IN ('Tarjeta', 'Transferencia')
GROUP BY Categoria_Producto;

```


* **Solución Punto 4:**
```sql
SELECT Categoria_Producto, AVG(Precio_Unitario) AS Precio_Promedio 
FROM ventas_retail 
WHERE Cantidad <= 3 
GROUP BY Categoria_Producto 
ORDER BY Precio_Promedio DESC 
LIMIT 1;

```




* **Solución Punto 5 (Python):**


In [ ]:
import sqlite3
import pandas as pd

# Conexión local a la ubicación física de la BD
ruta_db = r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\2_clases\unidad4\2_datos\monitoreo_y_gestion.db"
conexion = sqlite3.connect(ruta_db)

# Consulta del Punto 4
query = """
SELECT Categoria_Producto, AVG(Precio_Unitario) AS Precio_Promedio 
FROM ventas_retail 
WHERE Cantidad <= 3 
GROUP BY Categoria_Producto 
ORDER BY Precio_Promedio DESC 
LIMIT 1;
"""

df_lider_ventas = pd.read_sql_query(query, conexion)
print("=== CATEGORÍA LÍDER EN PRECIO PROMEDIO ===")
print(df_lider_ventas)

conexion.close()


# Asignación Nicolas  

Aquí tienes el diseño para el **Parcial N° 3 de Fundamentos de Bases de Datos (SQL)**, aplicado de manera exclusiva a la Tabla 3 (`uso_streaming`) de la base de datos relacional `monitoreo_y_gestion.db`.

Manteniendo la coherencia metodológica y la curva de complejidad incremental exigida por tu plan de evaluación, este parcial se enfoca en el área de **Analítica de Productos Digitales y Comportamiento del Usuario (*User Engagement & Retention*)**.

---



# PARCIAL 3: ANALÍTICA DE PRODUCTOS DIGITALES Y RETENCIÓN CON SQL

**Asignatura:** Fundamentos de Programación / Bases de Datos

**Profesor:** Marco Cañas

**Base de Datos Relacional:** `monitoreo_y_gestion.db`

**Tabla de Trabajo:** `uso_streaming`

---



### Contexto del Examen: Esquema de la Tabla

La tabla `uso_streaming` consolida la actividad mensual de los suscriptores y contiene las siguientes columnas:

* `ID_Usuario` (Texto)
* `Edad` (Decimal)
* `Tipo_Suscripcion` (Texto: Gratuito, Básico, Premium)
* `Minutos_Vistos_Mes` (Decimal)
* `Dispositivo_Principal` (Texto: Smartphone, Tablet, Smart TV, PC)
* `Cancelo_Suscripcion` (Texto: Sí, No)

---



### Punto 1: Proyección de Atributos de Cuentas de Usuario (10 Puntos)

**Sentencias a evaluar:** `SELECT` y `FROM`

**Enunciado:** El equipo de soporte técnico de la plataforma requiere un reporte básico para auditar la compatibilidad de los entornos de reproducción. Escribe una consulta en SQL que extraiga **únicamente** las columnas correspondientes al identificador del usuario, el tipo de suscripción que posee y su dispositivo principal de todos los registros de la tabla.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Al ejecutar esta consulta, notarás que las filas se muestran en el orden exacto en el que fueron insertadas en el archivo físico. ¿Garantiza una base de datos relacional este orden de manera permanente si no se incluye de forma explícita una cláusula de ordenamiento? Justifica tu respuesta.

---



### 📌 Punto 2: Segmentación de Clientes en Riesgo de Abandono (10 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM` y `WHERE`

**Enunciado:** El área de retención de clientes (*Customer Success*) busca diseñar una campaña de fidelización dirigida a usuarios jóvenes que están perdiendo el interés en la plataforma y podrían cancelar su cuenta. Escribe una consulta en SQL que devuelva el `ID_Usuario`, la `Edad` y los `Minutos_Vistos_Mes` **solo** para aquellos usuarios cuya `Edad` sea menor de 25 años **y** que registren un consumo de `Minutos_Vistos_Mes` inferior a 120.0 minutos, pero que **aún no** hayan cancelado la suscripción (`Cancelo_Suscripcion = 'No'`).

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Explica conceptualmente cómo procesa el motor SQL una cláusula `WHERE` que encadena múltiples condiciones mediante el operador `AND`. ¿Qué ocurre con la evaluación de una fila si la primera condición se cumple pero la segunda resulta falsa?

---



### 📌 Punto 3: Diagnóstico de Consumo por Tipo de Suscripción (15 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM`, `WHERE` y `GROUP BY`

**Enunciado:** El departamento de infraestructura de servidores necesita medir el tráfico de datos para optimizar el ancho de banda según los niveles de pago de la plataforma. Escribe una consulta en SQL que muestre el `Tipo_Suscripcion` y el **promedio de minutos vistos al mes** (utiliza la función de agregación `AVG(Minutos_Vistos_Mes)` renombrada como `Consumo_Promedio`) agrupado por cada tipo de plan. Restringe la consulta para evaluar **únicamente** a los usuarios que consumen el servicio desde pantallas grandes (`Smart TV` o `PC`).

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Si quisieras conocer el promedio de minutos consumidos únicamente para aquellas categorías de suscripción cuyo promedio final supere los 500 minutos, ¿podrías colocar esa restricción dentro del `WHERE` (ej. `WHERE AVG(Minutos_Vistos_Mes) > 500`)? Justifica técnicamente por qué se genera un error de sintaxis y qué cláusula se debería usar en su lugar.

---



# Punto 4: Identificación del Dispositivo Líder en Pérdida de Clientes (15 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `ORDER BY` y `LIMIT`

**Enunciado:** El equipo de desarrollo de aplicaciones (*App Developers*) sospecha que una de sus interfaces de usuario tiene fallas técnicas críticas que provocan altas tasas de deserción. Escribe una consulta en SQL que agrupe los datos por `Dispositivo_Principal` y calcule el **total de usuarios que abandonaron la plataforma** (utiliza la función `COUNT(ID_Usuario)` renombrada como `Total_Bajas`). Aplica los siguientes parámetros:

1. Filtra mediante `WHERE` para incluir únicamente a los registros donde el usuario efectivamente haya cancelado el servicio (`Cancelo_Suscripcion = 'Sí'`).
2. Ordena los resultados de manera descendente (`DESC`) según el conteo de bajas calculado.
3. Restringe el resultado utilizando `LIMIT` para extraer **únicamente el dispositivo que concentra el mayor número de cancelaciones**.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** El comando `LIMIT 1` se ejecuta al final de todo el flujo de procesamiento. Si dos dispositivos distintos compartieran exactamente la misma cantidad máxima de cancelaciones (un empate), ¿qué problema analítico se presentaría al usar `LIMIT 1` y qué estrategia de ordenamiento adicional implementarías en el `ORDER BY` para resolverlo de forma determinista?

---



### 📌 Punto 5: Implementación del Reporte en el Pipeline de Python (Bono de Clase)

**Enunciado:** Abre una celda de código en tu cuaderno de Jupyter y desarrolla el script de Python necesario para conectarte a la base de datos `monitoreo_y_gestion.db` (usando `sqlite3`). Carga el reporte estructurado en el **Punto 4** directamente a un DataFrame de Pandas llamado `df_fuga_dispositivo` e imprime el resultado final en pantalla.

* **Celda de Código Python:**

```python
# Desarrolla el bloque de código de extracción aquí

```

---



# Hoja de Respuestas Esperadas (Para uso exclusivo del Profesor):

* **Solución Punto 1:**
```sql
SELECT ID_Usuario, Tipo_Suscripcion, Dispositivo_Principal FROM uso_streaming;

```


* **Solución Punto 2:**
```sql
SELECT ID_Usuario, Edad, Minutos_Vistos_Mes 
FROM uso_streaming 
WHERE Edad < 25 
  AND Minutos_Vistos_Mes < 120.0 
  AND Cancelo_Suscripcion = 'No';

```


* **Solución Punto 3:**
```sql
SELECT Tipo_Suscripcion, AVG(Minutos_Vistos_Mes) AS Consumo_Promedio 
FROM uso_streaming 
WHERE Dispositivo_Principal = 'Smart TV' OR Dispositivo_Principal = 'PC' -- También es válido: WHERE Dispositivo_Principal IN ('Smart TV', 'PC')
GROUP BY Tipo_Suscripcion;

```


* **Solución Punto 4:**
```sql
SELECT Dispositivo_Principal, COUNT(ID_Usuario) AS Total_Bajas 
FROM uso_streaming 
WHERE Cancelo_Suscripcion = 'Sí' 
GROUP BY Dispositivo_Principal 
ORDER BY Total_Bajas DESC 
LIMIT 1;

```


* **Solución Punto 5 (Python):**
```python
import sqlite3
import pandas as pd

# Conexión a la base de datos centralizada
ruta_db = r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\2_clases\unidad4\2_datos\monitoreo_y_gestion.db"
conexion = sqlite3.connect(ruta_db)

# Consulta del Punto 4
query = """
SELECT Dispositivo_Principal, COUNT(ID_Usuario) AS Total_Bajas 
FROM uso_streaming 
WHERE Cancelo_Suscripcion = 'Sí' 
GROUP BY Dispositivo_Principal 
ORDER BY Total_Bajas DESC 
LIMIT 1;
"""

df_fuga_dispositivo = pd.read_sql_query(query, conexion)
print("=== PLATAFORMA CRÍTICA DE DESERCIÓN (CHURN) ===")
print(df_fuga_dispositivo)

conexion.close()

```

# Asignación Cristian  

Diseño para el **Parcial N° 4 de Fundamentos de Bases de Datos (SQL)**, aplicado de manera exclusiva a la Tabla 4 (`indicadores_salud`) de la base de datos relacional `monitoreo_y_gestion.db`.

Siguiendo tu esquema metodológico de complejidad acumulativa cláusula por cláusula, este examen traslada el contexto hacia el ámbito de la **Bioestadística y la Analítica de Datos Clínicos y Epidemiológicos**.




# PARCIAL 4: EXTRACCIÓN Y DIAGNÓSTICO DE DATOS CLÍNICOS CON SQL

**Asignatura:** Fundamentos de Programación / Bases de Datos

**Profesor:** Marco Cañas

**Base de Datos Relacional:** `monitoreo_y_gestion.db`

**Tabla de Trabajo:** `indicadores_salud`

---



### Contexto del Examen: Esquema de la Tabla

La tabla `indicadores_salud` consolida las métricas e historial de telemetría médica de los pacientes ingresados. Cuenta con las siguientes columnas numéricas y categóricas:

* `ID_Paciente` (Texto)
* `Edad` (Entero)
* `Presion_Sistolica` (Decimal: Medida en mm Hg)
* `Colesterol_mgDl` (Decimal: Medida en mg/dL)
* `Fumador` (Texto: Sí, No)
* `IMC` (Decimal: Índice de Masa Corporal)
* `Diagnostico_Riesgo` (Texto: Bajo, Moderado, Alto)

---



# Punto 1: Proyección de Atributos del Perfil Clínico (10 Puntos)

**Sentencias a evaluar:** `SELECT` y `FROM`

**Enunciado:** El área de enfermería general requiere generar un listado de control rápido para la ronda matutina. Escribe una consulta en SQL que extraiga **únicamente** las columnas correspondientes al identificador del paciente, su edad y su Índice de Masa Corporal (`IMC`) de todos los registros clínicos almacenados en la tabla.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Al escribir el comando `SELECT ID_Paciente, Edad, IMC`, ¿qué regla del modelo relacional determina el orden de las columnas en la tabla resultante? ¿Es posible alterar el orden físico original en el que fueron creadas las columnas del archivo mediante la sentencia `SELECT`? Justifica tu respuesta.

---



### 📌 Punto 2: Filtrado Epidemiológico de Pacientes Críticos (10 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM` y `WHERE`

**Enunciado:** El departamento de cardiología preventiva está diseñando un protocolo de intervención de urgencia. Escribe una consulta en SQL que devuelva el `ID_Paciente`, la `Presion_Sistolica` y el `Colesterol_mgDl` **solo** para aquellos pacientes que tengan un hábito de tabaquismo activo (`Fumador = 'Sí'`) **y** que simultáneamente registren una `Presion_Sistolica` estrictamente superior a 140.0 mm Hg (Hipertensión Grado 2).

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Si un registro clínico de la tabla posee una presión sistólica de 140.0 mm Hg exactos y el paciente es fumador, ¿aparecerá dicho paciente en el resultado de tu consulta? Justifica detalladamente basándote en el comportamiento de los operadores de comparación.

---

### 📌 Punto 3: Agregación Clínico-Cardiovascular por Estratos de Riesgo (15 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM`, `WHERE` y `GROUP BY`

**Enunciado:** La dirección epidemiológica del hospital desea evaluar si los niveles de lípidos en la sangre se corresponden de manera estadística con las alertas institucionales. Escribe una consulta en SQL que muestre la columna `Diagnostico_Riesgo` y el **promedio de colesterol** (utiliza la función de agregación `AVG(Colesterol_mgDl)` renombrada bajo el alias `Colesterol_Promedio`) agrupada por cada categoría de diagnóstico. Modifica la consulta para evaluar **únicamente** a los pacientes que sufren de sobrepeso u obesidad (es decir, aquellos con un `IMC` mayor o igual a 25.0).

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Explica por qué es metodológicamente incorrecto intentar aplicar el filtro del Índice de Masa Corporal utilizando la sentencia `HAVING` (ej. `HAVING IMC >= 25.0`) en lugar de usar `WHERE`. Establece la diferencia fundamental entre el momento en que actúa `WHERE` y el momento en que actúa `HAVING`.

---

### 📌 Punto 4: Identificación del Grupo de Máxima Alerta en Hipertensión (15 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `ORDER BY` y `LIMIT`

**Enunciado:** La junta médica necesita aislar estadísticamente el perfil del paciente que registra la mayor severidad de presión arterial sistólica entre los grupos con hábitos de riesgo consolidados. Escribe una consulta en SQL que agrupe los registros médicos por `Diagnostico_Riesgo` y calcule la **presión sistólica máxima** detectada (utiliza la función `MAX(Presion_Sistolica)` renombrada como `Maxima_Sistolica_Grupo`). Aplica los siguientes parámetros:

1. Filtra mediante `WHERE` para incluir únicamente a los pacientes que pertenecen al grupo de riesgo biológico de la tercera edad (`Edad >= 60`).
2. Ordena los resultados de manera descendente (`DESC`) según la presión máxima calculada.
3. Restringe la salida mediante la cláusula `LIMIT` para extraer **únicamente el grupo que posee el registro de presión más elevado**.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Describe detalladamente el ciclo de vida de los datos durante esta consulta. ¿Qué filas son descartadas en primera instancia en la memoria, sobre cuáles se realiza el cálculo de agrupación y cómo opera finalmente el ordenamiento junto al `LIMIT 1`?

---

### 📌 Punto 5: Extracción Automatizada para Análisis Bioestadístico en Jupyter (Bono)

**Enunciado:** Abre una celda de código en tu cuaderno de Jupyter y desarrolla el script de Python necesario para establecer conexión con la base de datos `monitoreo_y_gestion.db` (usando `sqlite3`). Envía la consulta estructurada en el **Punto 4** directamente a un DataFrame de Pandas llamado `df_alerta_cardiaca` e imprime el resultado final en pantalla.

* **Celda de Código Python:**

```python
# Desarrolla el bloque de código de extracción aquí

```

---



# Hoja de Respuestas Esperadas (Para uso exclusivo del Profesor):

* **Solución Punto 1:**
```sql
SELECT ID_Paciente, Edad, IMC FROM indicadores_salud;

```


* **Solución Punto 2:**
```sql
SELECT ID_Paciente, Presion_Sistolica, Colesterol_mgDl 
FROM indicadores_salud 
WHERE Fumador = 'Sí' 
  AND Presion_Sistolica > 140.0;

```


* **Solución Punto 3:**
```sql
SELECT Diagnostico_Riesgo, AVG(Colesterol_mgDl) AS Colesterol_Promedio 
FROM indicadores_salud 
WHERE IMC >= 25.0 
GROUP BY Diagnostico_Riesgo;

```


* **Solución Punto 4:**
```sql
SELECT Diagnostico_Riesgo, MAX(Presion_Sistolica) AS Maxima_Sistolica_Grupo 
FROM indicadores_salud 
WHERE Edad >= 60 
GROUP BY Diagnostico_Riesgo 
ORDER BY Maxima_Sistolica_Grupo DESC 
LIMIT 1;

```




* **Solución Punto 5 (Python):**


In [4]:
import sqlite3
import pandas as pd

# Conexión relacional local
ruta_db = r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\3_paciales\4_parcial\2_datos\monitoreo_y_gestion.db"
conexion = sqlite3.connect(ruta_db)

# REFORMA: Filtramos por IMC >= 25.0 en lugar de la columna Edad inexistente
query = """
SELECT Diagnostico_Riesgo, MAX(Presion_Sistolica) AS Maxima_Sistolica_Grupo 
FROM indicadores_salud 
WHERE IMC >= 25.0 
GROUP BY Diagnostico_Riesgo 
ORDER BY Maxima_Sistolica_Grupo DESC 
LIMIT 1;
"""

try:
    df_alerta_cardiaca = pd.read_sql_query(query, conexion)
    print("=== PROTOCOLO DE ALERTA EPIDEMIOLÓGICA MÁXIMA ===")
    print(df_alerta_cardiaca)
except pd.errors.DatabaseError as e:
    print(f"Error al ejecutar la consulta: {e}")

conexion.close()

=== PROTOCOLO DE ALERTA EPIDEMIOLÓGICA MÁXIMA ===
  Diagnostico_Riesgo  Maxima_Sistolica_Grupo
0               Bajo                   179.0


# Asignación Alexander  

Aquí tienes el diseño para el **Parcial N° 5 de Fundamentos de Bases de Datos (SQL)**, aplicado de manera exclusiva a la Tabla 5 (`desempeño_rrhh`) de la base de datos relacional `monitoreo_y_gestion.db`.

Cerrando este ciclo de evaluaciones y respetando la estructura de complejidad acumulativa cláusula por cláusula, el examen se ambienta en el campo corporativo de **Analítica de Recursos Humanos y Gestión del Talento (*People Analytics*)**.

---



# PARCIAL 5: ANALÍTICA DE RECURSOS HUMANOS Y PEOPLE ANALYTICS CON SQL

**Asignatura:** Fundamentos de Programación / Bases de Datos

**Profesor:** Marco Cañas

**Base de Datos Relacional:** `monitoreo_y_gestion.db`

**Tabla de Trabajo:** `desempeño_rrhh`

---



### Contexto del Examen: Esquema de la Tabla

La tabla `desempeño_rrhh` consolida la estructura de la plantilla de empleados y sus métricas anuales de rendimiento. Cuenta con las siguientes columnas:

* `ID_Empleado` (Texto)
* `Departamento` (Texto: TI, Ventas, RRHH, Finanzas, Operaciones)
* `Anos_Experiencia` (Entero)
* `Salario_Mensual_USD` (Decimal)
* `Puntuacion_Evaluacion` (Decimal: Escala de 1.0 a 5.0)
* `Modalidad` (Texto: Presencial, Remoto, Híbrido)

---



# Punto 1: Proyección de Atributos de la Nómina (10 Puntos)

**Sentencias a evaluar:** `SELECT` y `FROM`

**Enunciado:** El departamento de nómina y compensaciones requiere exportar una lista básica para auditar los salarios actuales de la compañía. Escribe una consulta en SQL que extraiga **únicamente** las columnas correspondientes al identificador del empleado, su departamento y su salario mensual en dólares de todos los registros de la tabla.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Al ejecutar el comando, notarás que las columnas aparecen exactamente en el orden en que las escribiste en el `SELECT`. Si un analista de datos necesita que en el reporte visual aparezca primero el salario y luego el departamento, ¿se debe modificar el archivo físico de la base de datos o basta con reordenar los campos en la sentencia de proyección? Explica el concepto de independencia de datos.

---



### 📌 Punto 2: Filtrado de Perfiles Clínicos de Alto Rendimiento Remoto (10 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM` y `WHERE`

**Enunciado:** La dirección general está buscando candidatos internos calificados para liderar un nuevo proyecto de transformación digital que operará bajo teletrabajo. Escribe una consulta en SQL que devuelva el `ID_Empleado`, los `Anos_Experiencia` y la `Puntuacion_Evaluacion` **solo** para aquellos colaboradores que trabajen en la `Modalidad` 'Remoto' **y** que tengan una `Puntuacion_Evaluacion` estrictamente superior a 4.5.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** ¿Qué ocurriría con el universo de empleados devuelto si cambias el operador lógico `AND` por un `OR`? Describe el impacto que tendría este cambio en la toma de decisiones del equipo de Recursos Humanos si están buscando perfiles de excelencia que cumplan *ambos* requisitos.

---



### 📌 Punto 3: Análisis del Costo Salarial por Departamentos (15 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM`, `WHERE` y `GROUP BY`

**Enunciado:** El comité financiero necesita evaluar el presupuesto operativo por departamentos para el próximo año, enfocándose exclusivamente en el personal consolidado. Escribe una consulta en SQL que muestre la columna `Departamento` y el **salario mensual promedio** (utiliza la función de agregación `AVG(Salario_Mensual_USD)` renombrada bajo el alias `Salario_Promedio`) agrupada por cada departamento. Restringe la consulta para evaluar **únicamente** al personal senior o con trayectoria (aquellos empleados con `Anos_Experiencia` mayor o igual a 5 años).

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Imaginemos que un estudiante intenta escribir la consulta agregando la cláusula `WHERE Salario_Promedio > 5000` para filtrar los departamentos costosos. Explica técnicamente por qué el motor SQL arrojaría un error al intentar usar un alias o una función de agregación dentro del `WHERE` y qué sentencia se debió utilizar para resolver ese filtrado posterior.

---



### 📌 Punto 4: Identificación del Área con Menor Rendimiento (15 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `ORDER BY` y `LIMIT`

**Enunciado:** El equipo de desarrollo organizacional planea diseñar un programa intensivo de capacitación y mentorías dirigido al departamento que registre las métricas de desempeño más preocupantes en el personal operativo. Escribe una consulta en SQL que agrupe los registros de los empleados por `Departamento` y calcule la **puntuación de evaluación promedio** (utiliza la función `AVG(Puntuacion_Evaluacion)` renombrada como `Rendimiento_Promedio`). Aplica los siguientes parámetros:

1. Filtra mediante `WHERE` para incluir únicamente al personal que trabaja bajo la modalidad 'Presencial' (evaluación de infraestructura física).
2. Ordena los resultados de manera **ascendente** (`ASC`) según el rendimiento promedio calculado (para que el promedio más bajo quede en primer lugar).
3. Restringe la salida mediante la cláusula `LIMIT` para extraer **únicamente el departamento con el rendimiento promedio más bajo**.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Explica detalladamente por qué el ordenamiento de los resultados (`ORDER BY`) y la restricción de filas (`LIMIT`) se ejecutan al final del ciclo de procesamiento de la base de datos. ¿Qué problemas de rendimiento ocurrirían si el motor intentara aplicar un `LIMIT` antes de agrupar los datos con el `GROUP BY`?

---

### 📌 Punto 5: Conexión de People Analytics mediante Pandas (Bono de Clase)

**Enunciado:** Abre una celda de código en tu cuaderno de Jupyter y desarrolla el script de Python necesario para establecer conexión con la base de datos `monitoreo_y_gestion.db` (utilizando la librería nativa `sqlite3`). Envía la consulta formulada en el **Punto 4** directamente a un DataFrame de Pandas llamado `df_urgencia_capacitacion` e imprime el resultado final en la pantalla.

* **Celda de Código Python:**

```python
# Desarrolla el bloque de código de extracción aquí

```

---



# Hoja de Respuestas Esperadas (Para uso exclusivo del Profesor):

* **Solución Punto 1:**
```sql
SELECT ID_Empleado, Departamento, Salario_Mensual_USD FROM desempeño_rrhh;

```


* **Solución Punto 2:**
```sql
SELECT ID_Empleado, Anos_Experiencia, Puntuacion_Evaluacion 
FROM desempeño_rrhh 
WHERE Modalidad = 'Remoto' 
  AND Puntuacion_Evaluacion > 4.5;

```


* **Solución Punto 3:**
```sql
SELECT Departamento, AVG(Salario_Mensual_USD) AS Salario_Promedio 
FROM desempeño_rrhh 
WHERE Anos_Experiencia >= 5 
GROUP BY Departamento;

```


* **Solución Punto 4:**
```sql
SELECT Departamento, AVG(Puntuacion_Evaluacion) AS Rendimiento_Promedio 
FROM desempeño_rrhh 
WHERE Modalidad = 'Presencial' 
GROUP BY Departamento 
ORDER BY Rendimiento_Promedio ASC 
LIMIT 1;

```


* **Solución Punto 5 (Python):**
```python


In [6]:
import sqlite3
import pandas as pd

# Conexión local a la base de datos relacional
ruta_db = r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\3_paciales\4_parcial\2_datos\monitoreo_y_gestion.db"
conexion = sqlite3.connect(ruta_db)

# Consulta estructurada en el Punto 4
query = """
SELECT Departamento, AVG(Puntuacion_Evaluacion) AS Rendimiento_Promedio 
FROM desempeño_rrhh 
WHERE Modalidad = 'Presencial' 
GROUP BY Departamento 
ORDER BY Rendimiento_Promedio ASC 
LIMIT 1;
"""

df_urgencia_capacitacion = pd.read_sql_query(query, conexion)
print("=== DEPARTAMENTO PRIORITARIO PARA CAPACITACIÓN ===")
print(df_urgencia_capacitacion)

conexion.close()


=== DEPARTAMENTO PRIORITARIO PARA CAPACITACIÓN ===
       Departamento  Rendimiento_Promedio
0  Recursos Humanos                   2.3


# Asigmación Santiago  



# PARCIAL 6: ANALÍTICA CLIMATOLÓGICA Y MONITOREO AMBIENTAL CON SQL

**Asignatura:** Fundamentos de Programación / Bases de Datos

**Profesor:** Marco Cañas

**Base de Datos Relacional:** `monitoreo_y_gestion.db`

**Tabla de Trabajo:** `contaminacion_ambiental`

---



# Contexto del Examen: Esquema de la Tabla

La tabla `contaminacion_ambiental` consolida las lecturas diarias de telemetría de diversas estaciones de monitoreo urbano. Cuenta con las siguientes columnas:

* `ID_Registro` (Texto)
* `Ciudad` (Texto: Bogotá, Medellín, Cali, Barranquilla, Bucaramanga)
* `Nivel_PM25` (Decimal: Concentración de material particulado fino en $\mu g/m^3$)
* `Temperatura_C` (Decimal: Temperatura en grados Celsius)
* `Humedad_Porcentaje` (Decimal: Porcentaje de humedad relativa)
* `Clasificacion_Calidad` (Texto: Buena, Moderada, Dañina)

---



# Punto 1: Proyección de Atributos de Telemetría (10 Puntos)

**Sentencias a evaluar:** `SELECT` y `FROM`

**Enunciado:** El comité técnico del Ministerio de Ambiente requiere un reporte básico de las condiciones higrotérmicas de las estaciones para calibrar los sensores. Escribe una consulta en SQL que extraiga **únicamente** las columnas correspondientes a la ciudad, la temperatura en grados Celsius y el porcentaje de humedad de todos los registros de la tabla.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Al ejecutar `SELECT Ciudad, Temperatura_C, Humedad_Porcentaje FROM contaminacion_ambiental;`, el motor nos devuelve una matriz de datos. Si la tabla no cuenta con una Clave Primaria (*Primary Key*) explícita, ¿se puede garantizar que dos filas de la consulta no sean exactamente idénticas? Explica el concepto de tuplas duplicadas en el modelo relacional.

---



# Punto 2: Filtrado de Alertas de Contaminación Crítica (10 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM` y `WHERE`

**Enunciado:** El sistema de Alertas Tempranas Ambientales necesita aislar los registros donde la calidad del aire representa un peligro inminente para la salud de la población vulnerable. Escribe una consulta en SQL que devuelva el `ID_Registro`, la `Ciudad` y el `Nivel_PM25` **solo** para aquellos días donde la `Clasificacion_Calidad` sea igual a 'Dañina' **y** el `Nivel_PM25` sea estrictamente superior a 55.0 $\mu g/m^3$.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** En bases de datos, los valores de tipo texto (cadenas de caracteres) son sensibles a mayúsculas, minúsculas y acentos según la configuración (*Collation*). Si en la base de datos la palabra está guardada como 'Dañina' y en tu cláusula `WHERE` escribes `Clasificacion_Calidad = 'dañina'`, ¿qué problema potencial podría ocurrir en el filtrado?

---



# Punto 3: Caracterización del Aire por Centros Urbanos (15 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM`, `WHERE` y `GROUP BY`

**Enunciado:** La Secretaría de Salud Pública está realizando un estudio epidemiológico comparativo y requiere conocer la concentración promedio de partículas finas por región. Escribe una consulta en SQL que muestre la columna `Ciudad` y el **promedio del nivel de PM2.5** (utiliza la función de agregación `AVG(Nivel_PM25)` renombrada bajo el alias `PM25_Promedio`) agrupada por cada ciudad. Restringe la consulta para evaluar **únicamente** los días calurosos (aquellos donde la `Temperatura_C` sea mayor o igual a 28.0).

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Describe la diferencia entre el funcionamiento de las funciones de agregación (como `AVG` o `SUM`) y las funciones escalares. ¿Por qué no es posible mezclar libremente una columna no agrupada con una función de agregación en el `SELECT` sin recurrir a la sentencia `GROUP BY`?




# Punto 4: Identificación de la Ciudad con Mayor Sequedad Atmosférica (15 Puntos)

**Sentencias a evaluar:** `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `ORDER BY` y `LIMIT`

**Enunciado:** El cuerpo de bomberos forestales necesita identificar qué zona urbana presenta las condiciones de menor humedad relativa promedio, lo que incrementa el riesgo de propagación de incendios. Escribe una consulta en SQL que agrupe los datos por `Ciudad` y calcule el **porcentaje de humedad promedio** (utiliza la función `AVG(Humedad_Porcentaje)` renombrada como `Humedad_Promedio`). Aplica los siguientes parámetros:

1. Filtra mediante `WHERE` para incluir únicamente los días donde los índices de contaminación fueron aceptables (`Clasificacion_Calidad = 'Buena'`).
2. Ordena los resultados de manera **ascendente** (`ASC`) según la humedad promedio calculada (para que el promedio más seco/bajo quede en primer lugar).
3. Restringe la salida mediante la cláusula `LIMIT` para extraer **únicamente la ciudad con el promedio de humedad más bajo**.

* **Celda de Código SQL:**

```sql
-- Escribe tu consulta aquí

```

* **Celda Markdown (Pregunta de Análisis):** Plantea el escenario físico de la consulta. Si el motor de la base de datos ejecutara la cláusula `LIMIT 1` antes de realizar el ordenamiento `ORDER BY`, ¿el resultado devuelto seguiría siendo el promedio de humedad más bajo de toda la tabla? Justifica por qué el orden jerárquico de las sentencias altera el sentido analítico del reporte.

---



# Punto 5: Extracción y Consolidación de Datos Ambientales en Python (Bono)

**Enunciado:** Abre una celda de código en tu cuaderno de Jupyter y desarrolla el script de Python necesario para establecer conexión con la base de datos `monitoreo_y_gestion.db` (utilizando la librería nativa `sqlite3`). Carga la consulta estructurada en el **Punto 4** directamente a un DataFrame de Pandas llamado `df_riesgo_incendio` e imprime el resultado en la pantalla.

* **Celda de Código Python:**

```python
# Desarrolla el bloque de código de extracción aquí

```

---



# Hoja de Respuestas Esperadas (Para uso exclusivo del Profesor):

* **Solución Punto 1:**
```sql
SELECT Ciudad, Temperatura_C, Humedad_Porcentaje FROM contaminacion_ambiental;

```


* **Solución Punto 2:**
```sql
SELECT ID_Registro, Ciudad, Nivel_PM25 
FROM contaminacion_ambiental 
WHERE Clasificacion_Calidad = 'Dañina' 
  AND Nivel_PM25 > 55.0;

```


* **Solución Punto 3:**
```sql
SELECT Ciudad, AVG(Nivel_PM25) AS PM25_Promedio 
FROM contaminacion_ambiental 
WHERE Temperatura_C >= 28.0 
GROUP BY Ciudad;

```


* **Solución Punto 4:**
```sql
SELECT Ciudad, AVG(Humedad_Porcentaje) AS Humedad_Promedio 
FROM contaminacion_ambiental 
WHERE Clasificacion_Calidad = 'Buena' 
GROUP BY Ciudad 
ORDER BY Humedad_Promedio ASC 
LIMIT 1;

```




* **Solución Punto 5 (Python):**


In [7]:
import sqlite3
import pandas as pd

# Conexión local a la base de datos centralizada
ruta_db = r"C:\Users\marco\Documentos\docencia\fundamentos_de_programacion\3_paciales\4_parcial\2_datos\monitoreo_y_gestion.db"
conexion = sqlite3.connect(ruta_db)

# Consulta estructurada en el Punto 4
query = """
SELECT Ciudad, AVG(Humedad_Porcentaje) AS Humedad_Promedio 
FROM contaminacion_ambiental 
WHERE Clasificacion_Calidad = 'Buena' 
GROUP BY Ciudad 
ORDER BY Humedad_Promedio ASC 
LIMIT 1;
"""

df_riesgo_incendio = pd.read_sql_query(query, conexion)
print("=== URBANO CRÍTICO: MÁXIMO RIESGO DE INCENDIO ===")
print(df_riesgo_incendio)

conexion.close()


=== URBANO CRÍTICO: MÁXIMO RIESGO DE INCENDIO ===
  Ciudad  Humedad_Promedio
0   Cali         61.636364
